In [26]:
# revT2V: compare reverse-time methods
#
# All methods run on one shared U-Net (revt2v.infer.MethodBank) from the
# same initial noise z:
#   teacher        - frozen ModelScope teacher, forward time, from z
#   matched_target - reverse(teacher(latents = flip(z))), the per-sample
#                    reference a reverse generator started from z should
#                    reproduce
#   attn_injection - capture the base U-Net's own attention on flip(x_t)
#                    (stock attention, no LoRA), inject it rotated 180
#                    degrees against the real pass's own V/to_out -- a
#                    trained LoRA on to_v/to_out.0 is used if its checkpoint
#                    exists, otherwise this runs zero-shot
#   conv_oracle    - teacher with every temporal Conv3d kernel flipped in
#                    time; no training
#
# Caveat: conv_oracle is an exact time-mirror of the teacher
# (unet_flipped(z) == flip(unet(flip(z)))), so it is equivalent to reversing
# the teacher's output. It's a reference / upper bound, not a learned method.
#
# Scoring is done by scripts/evaluate.py, every method vs matched_target:
#   FVD            - I3D features vs the matched-target set (lower =
#                    better). Noisy with few videos, so directional only.
#   CLIP score     - 100 x CLIP cosine(frame, prompt), averaged over frames
#                    (higher = better).
#   VBench dynamic - binary per video; per method, the fraction of videos
#   degree           that move.
#   Flow cosine    - (RAFT, Farneback fallback, at 256x256) direction
#                    agreement of the signed flow vs the target at the same
#                    frame pair (+1 = same motion direction, -1 = opposite).
#
# matched_target is the reference, so its FVD and flow cosine are NaN. The
# summary averages each per-video metric per method.

# 1. Setup
# Change this to your GitHub username
GITHUB_USER = "silentlooop"

!git clone -q https://github.com/{GITHUB_USER}/revT2V.git /content/revT2V 2>/dev/null || git -C /content/revT2V pull -q
%cd /content/revT2V
!pip install -q -e ".[eval]"
# VBench pins transformers==4.33.2 and numpy<2, which would break diffusers, so
# install it without deps plus easydict, the only extra its dynamic_degree needs.
# decord has no wheels for Python 3.13; scripts/evaluate.py swaps in an imageio stand-in.
!pip install -q --no-deps vbench
!pip install -q easydict
!pip uninstall torchao -y
# no %autoreload: it breaks on Colab's Python 3.13 (removed `imp` module).
# After a git pull, restart the kernel and re-run from the top.

/content/revT2V
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for revt2v (pyproject.toml) ... done


In [27]:
import os
import torch

HF_TOKEN = ""  # paste your Hugging Face token here
os.environ["HF_TOKEN"] = HF_TOKEN
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

CUDA: True | Tesla T4


In [28]:
HF_REPO = "silentlooop/revt2v-ckpt"

PROMPT_SETS = {
    "physics": [
        "A woman walking away along a beach, back view, static camera",
        "orange juice being poured into a glass, close-up",
        "a man walking away from the camera down a long corridor, static camera",
        "Slow motion of a soap bubble popping, black background",
        "Time lapse of a candle melting"
    ]
}
PROMPT_SET = "physics"  # "physics" | "irreversible"
PROMPTS = PROMPT_SETS[PROMPT_SET]
OUT_DIR = f"/content/compare_results/{PROMPT_SET}"  # one folder per set, so runs don't overwrite
VIDEOS_DIR = f"{OUT_DIR}/videos"  # <method>/<name>.mp4 + prompts.json, the layout evaluate.py scores
SEEDS = [0, 1]
GEN = dict(negative_prompt="watermark, text", num_inference_steps=25, guidance_scale=9.0,
           num_frames=16, height=256, width=256)
SHEET_FRAMES = [0, 5, 10, 15]
FPS = 8
os.makedirs(OUT_DIR, exist_ok=True)

In [29]:
# 2. Load the teacher once, then the trained adapters
from revt2v.infer import MethodBank

bank = MethodBank(hf_repo_id=HF_REPO)
bank.pipeline.set_progress_bar_config(disable=True)

# attn_injection runs zero-shot if its checkpoint doesn't exist yet (it's
# always in bank.methods) -- the others are skipped entirely without one.
for method in ["conv_lora", "attn_injection"]:
    try:
        print(method, "loaded" if bank.load_adapter(method) else "-> no checkpoint, skipped")
    except Exception as exc:  # e.g. root checkpoint belongs to another method
        print(method, "-> skipped:", exc)

print("available methods:", bank.methods)

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


conv_lora loaded
attn_injection -> skipped: 'attn_injection'
available methods: ['teacher', 'conv_oracle', 'conv_lora']


In [30]:
def initial_noise(seed):
    g = torch.Generator(device="cuda").manual_seed(seed)
    return torch.randn((1, 4, GEN["num_frames"], GEN["height"] // 8, GEN["width"] // 8),
                       generator=g, device="cuda", dtype=bank.pipeline.unet.dtype)

def to_np(video):
    """(F, H, W, C) float tensor in [0, 1] -> float32 numpy."""
    return video.float().clamp(0, 1).cpu().numpy()

In [31]:
# 3. Exactness checks
# (a) U-Net level: unet_flipped(z, t, emb) vs flip(unet(flip(z), t, emb))
from contextlib import nullcontext
from revt2v.methods.conv_mirror import find_temporal_convs, flip_temporal_convs, set_flip_state
from revt2v.teacher import encode_prompt

unet = bank.pipeline.unet
print(len(find_temporal_convs(unet)), "temporal Conv3d layers")

z = initial_noise(0)
emb, _ = encode_prompt(bank.pipeline, PROMPTS[0])
emb = emb.to(unet.dtype)
t = torch.tensor([500], device="cuda")

adapters_off = bank.student.disable_adapter() if bank.student is not None else nullcontext()
try:
    with torch.no_grad(), adapters_off:
        mirrored = unet(z.flip(2), t, encoder_hidden_states=emb, return_dict=False)[0].flip(2)
        flip_temporal_convs(unet)
        flipped = unet(z, t, encoder_hidden_states=emb, return_dict=False)[0]
finally:
    set_flip_state(unet, ())

print(f"max |unet_flipped(z) - flip(unet(flip(z)))| = {(flipped.float() - mirrored.float()).abs().max().item():.3e}")

88 temporal Conv3d layers
max |unet_flipped(z) - flip(unet(flip(z)))| = 4.639e-03


In [32]:
# (b) Video level: conv_oracle(latents=z) vs matched_target(prompt, z)
z = initial_noise(0)
oracle = to_np(bank.generate(PROMPTS[0], "conv_oracle", latents=z, **GEN)["video"])
target = to_np(bank.matched_target(PROMPTS[0], z, **GEN)["video"])
print(f"mean pixel diff (0-1 scale): {abs(oracle - target).mean():.4f}")

mean pixel diff (0-1 scale): 0.0124


In [33]:
# 4. Load scripts/evaluate.py
import sys

import numpy as np

# scripts/ isn't a package (and `evaluate` would clash with Hugging Face's
# `evaluate` library) -- same sys.path trick server/app.py uses to import it.
sys.path.insert(0, "scripts")
import evaluate as ev

In [34]:
# 5. Generate every method from the same noise
from diffusers.utils import export_to_video
from IPython.display import Video, display

COLUMNS = [m for m in ["teacher", "matched_target", "attn_injection", "conv_oracle",
           "conv_lora"]
           if m == "matched_target" or m in bank.methods]

# Every method + matched_target from the same noise per (prompt, seed); names are p<idx>_s<seed>.
videos, names = ev.generate_videos(bank, PROMPTS, SEEDS, [m for m in COLUMNS if m != "matched_target"], GEN)
ev.save_video_set(videos, names, VIDEOS_DIR, fps=FPS)

os.makedirs(f"{OUT_DIR}/side_by_side", exist_ok=True)
for name, prompt in names.items():
    side_by_side = np.clip(np.concatenate([videos[m][name] for m in COLUMNS], axis=2), 0, 1)
    # export_to_video expects float frames in [0, 1], not uint8
    path = export_to_video(list(side_by_side), f"{OUT_DIR}/side_by_side/{name}.mp4", fps=FPS)
    print(f"{name}: {prompt}\n    " + " | ".join(COLUMNS))
    display(Video(path, embed=True, width=256 * len(COLUMNS)))

p0_s0: A woman walking away along a beach, back view, static camera
    teacher | matched_target | conv_oracle | conv_lora


p0_s1: A woman walking away along a beach, back view, static camera
    teacher | matched_target | conv_oracle | conv_lora


p1_s0: orange juice being poured into a glass, close-up
    teacher | matched_target | conv_oracle | conv_lora


p1_s1: orange juice being poured into a glass, close-up
    teacher | matched_target | conv_oracle | conv_lora


p2_s0: a man walking away from the camera down a long corridor, static camera
    teacher | matched_target | conv_oracle | conv_lora


p2_s1: a man walking away from the camera down a long corridor, static camera
    teacher | matched_target | conv_oracle | conv_lora


p3_s0: Slow motion of a soap bubble popping, black background
    teacher | matched_target | conv_oracle | conv_lora


p3_s1: Slow motion of a soap bubble popping, black background
    teacher | matched_target | conv_oracle | conv_lora


p4_s0: Time lapse of a candle melting
    teacher | matched_target | conv_oracle | conv_lora


p4_s1: Time lapse of a candle melting
    teacher | matched_target | conv_oracle | conv_lora


In [35]:
# 6. Scores from scripts/evaluate.py (vs matched target)
import pandas as pd

# The first run downloads CLIP ViT-L/14, I3D and the VBench models (a few GB).
# Writes metrics.csv, summary.csv and summary.json to OUT_DIR.
result = ev.score_videos_dir(VIDEOS_DIR, output_dir=OUT_DIR)

summary = pd.DataFrame(result["summary"]).set_index("method")
display(summary.round(3))
display(pd.DataFrame(result["rows"]).round(3))

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Evaluation meta data saved to /content/compare_results/physics/vbench/conv_lora__dynamic_degree_full_info.json
cur_full_info_path: /content/compare_results/physics/vbench/conv_lora__dynamic_degree_full_info.json


  0%|          | 0/10 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/vbench/third_party/RAFT/core/raft.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/usr/local/lib/python3.13/dist-packages/vbench/third_party/RAFT/core/raft.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/usr/local/lib/python3.13/dist-packages/vbench/third_party/RAFT/core/raft.py:127: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
100%|██████████| 10/10 [00:16<00:00,  1.67s/it]


Evaluation results saved to /content/compare_results/physics/vbench/conv_lora__dynamic_degree_eval_results.json
Evaluation meta data saved to /content/compare_results/physics/vbench/conv_oracle__dynamic_degree_full_info.json
cur_full_info_path: /content/compare_results/physics/vbench/conv_oracle__dynamic_degree_full_info.json


100%|██████████| 10/10 [00:16<00:00,  1.67s/it]


...

Evaluation results saved to /content/compare_results/physics/vbench/matched_target__dynamic_degree_eval_results.json
Evaluation meta data saved to /content/compare_results/physics/vbench/teacher__dynamic_degree_full_info.json
cur_full_info_path: /content/compare_results/physics/vbench/teacher__dynamic_degree_full_info.json


100%|██████████| 10/10 [00:16<00:00,  1.66s/it]


Evaluation results saved to /content/compare_results/physics/vbench/teacher__dynamic_degree_eval_results.json


,videos,fvd,clip_score,flow_cosine,vbench_dynamic_degree
method,,,,,
conv_lora,10,5641.147,25.794,0.188,0.4
conv_oracle,10,93.623,27.451,0.951,0.7
matched_target,10,NaN,27.554,NaN,0.7
teacher,10,1406.969,28.369,0.430,0.6


,method,video,prompt,clip_score,flow_cosine,vbench_dynamic_degree
0,conv_lora,p0_s0,"A woman walking away along a beach, back view,...",27.998,0.059,1.0
1,conv_lora,p0_s1,"A woman walking away along a beach, back view,...",25.991,0.322,1.0
2,conv_lora,p1_s0,"orange juice being poured into a glass, close-up",30.181,0.233,0.0
3,conv_lora,p1_s1,"orange juice being poured into a glass, close-up",34.317,-0.037,1.0
4,conv_lora,p2_s0,a man walking away from the camera down a long...,20.560,-0.203,1.0
5,conv_lora,p2_s1,a man walking away from the camera down a long...,21.361,0.254,0.0
6,conv_lora,p3_s0,"Slow motion of a soap bubble popping, black ba...",22.677,0.037,0.0
7,conv_lora,p3_s1,"Slow motion of a soap bubble popping, black ba...",22.276,0.491,0.0
8,conv_lora,p4_s0,Time lapse of a candle melting,22.922,-0.048,0.0
9,conv_lora,p4_s1,Time lapse of a candle melting,29.653,0.773,0.0
